# Flang Bounds Sanitizer - Single-Click Build

**Note:** To ensure you are using the GPU, go to the top menu and click **Runtime -> Change runtime type** and select **T4 GPU** (or any available GPU). 

Run the single block below. It will execute everything sequentially and stream the live compilation progress directly to the output!

In [ ]:
%%bash
set -e # Exit immediately if any command fails

echo "=================================================="
echo "[1/6] Installing dependencies..."
echo "=================================================="
apt-get update -qq && apt-get install -y cmake ninja-build build-essential -qq

echo "=================================================="
echo "[2/6] Downloading LLVM & Custom Pass (this takes a few minutes)..."
echo "=================================================="
rm -rf llvm-project project
git clone --depth 1 --progress https://github.com/llvm/llvm-project.git
git clone --progress https://github.com/vishwapanchal/Flang-Bounds-Sanitizer.git project

echo "=================================================="
echo "[3/6] Integrating custom pass into LLVM..."
echo "=================================================="
cp project/src/pass/BoundsCheckInstrumentation.* llvm-project/flang/lib/Optimizer/Transforms/
sed -i '/add_flang_library(FIRTransforms/a \  BoundsCheckInstrumentation.cpp' llvm-project/flang/lib/Optimizer/Transforms/CMakeLists.txt

RUNTIME_DIR=$(find llvm-project -type d -path "*/flang-rt/lib/runtime" -o -path "*/flang/runtime" | head -n 1)
if [ -n "$RUNTIME_DIR" ]; then
  cp project/src/runtime/bounds-check.* "$RUNTIME_DIR/"
  if [[ "$RUNTIME_DIR" == *"flang-rt"* ]]; then
    sed -i '/add_flangrt_library(flang_rt.runtime/a \  bounds-check.cpp' "$RUNTIME_DIR/CMakeLists.txt"
  else
    sed -i '/add_flang_library(FlangRuntime/a \  bounds-check.cpp' "$RUNTIME_DIR/CMakeLists.txt"
  fi
fi

sed -i '/pm.addPass(hlfir::createLowerHLFIRIntrinsics());/i \  pm.addPass(fir::createHLFIRBoundsCheckPass());' llvm-project/flang/lib/Optimizer/Passes/Pipelines.cpp

echo "=================================================="
echo "[4/6] Configuring CMake (Release Mode for maximum speed)..."
echo "=================================================="
cd llvm-project
mkdir -p build && cd build
cmake -G Ninja ../llvm \
  -DCMAKE_BUILD_TYPE=Release \
  -DLLVM_ENABLE_PROJECTS="flang;mlir" \
  -DLLVM_ENABLE_RUNTIMES="flang-rt" \
  -DLLVM_TARGETS_TO_BUILD="X86" \
  -DLLVM_BUILD_TESTS=OFF \
  -DLLVM_BUILD_EXAMPLES=OFF \
  -DLLVM_INCLUDE_TESTS=OFF \
  -DLLVM_INCLUDE_EXAMPLES=OFF

echo "=================================================="
echo "[5/6] Building Flang (LIVE PROGRESS!)..."
echo "=================================================="
# Ninja automatically parallelizes the build to use all available CPU cores
ninja flang flang-rt || ninja flang FlangRuntime || ninja flang

echo "=================================================="
echo "[6/6] Testing the Compiler Output..."
echo "=================================================="
bin/flang -fc1 -emit-hlfir ../../project/src/tests/bounds_check.f90 -o bounds_check.hlfir
echo ""
echo "Found our injected bounds check function in the compiler output:"
cat bounds_check.hlfir | grep "_FortranABoundsCheck"
echo ""
echo "✅ SUCCESS! Compilation and integration complete!"
